In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob
import random
from functools import partial

from JSAnimation import IPython_display
from matplotlib import animation
from IPython.display import HTML
from math import log,exp

In [ ]:

portraits = glob.glob("images/portraits/*.jpg") + glob.glob("images/portraits/*.png")


def load_image(index=None):
    if not index:
        index = random.sample(range(len(portraits)), 1)[0]
    
    im_path = portraits[index]
    im_frame = Image.open(im_path).convert("L")
    np_frame = np.array(im_frame.getdata())#/256
    width, height = im_frame.size
    x = np_frame.reshape(height, width)
    return x, width, height



In [ ]:
def plot_face(port, ax=None):
    global width, height 
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(port.reshape(height, width), cmap=plt.cm.gray)
    ax.axis('off')
x, width, height = load_image(4)
x.shape
plot_face(x)

In [ ]:
def svd(X):
    U, sigma, V_t = np.linalg.svd(X, full_matrices=False)
    Sigma = np.diag(sigma)
    return U, Sigma, V_t
U, Sigma, V_t = svd(x)
U.shape, Sigma.shape, V_t.shape

In [ ]:
def truncated_svd(X, r):
    U, Sigma, V_t = svd(X)
    return U[:, :r], Sigma[:r, :r], V_t[:r,:]
U_r, Sigma_rr, V_t_r = truncated_svd(x, 6)
U_r.shape, Sigma_rr.shape, V_t_r.shape

In [ ]:

from tempfile import TemporaryDirectory 
from pathlib import Path
class NewFuncAnimation(animation.FuncAnimation):
    """
    This class overrides `to_jshtml` so it can pass the arguments to the save function.
    """
    def to_jshtml(self, fps=None, embed_frames=True, default_mode=None,  *args, **kwargs):
        """
        Generate HTML representation of the animation.

        Parameters
        ----------
        fps : int, optional
            Movie frame rate (per second). If not set, the frame rate from
            the animation's frame interval.
        embed_frames : bool, optional
        default_mode : str, optional
            What to do when the animation ends. Must be one of ``{'loop',
            'once', 'reflect'}``. Defaults to ``'loop'`` if the *repeat*
            parameter is True, otherwise ``'once'``.

        Returns
        -------
        str
            An HTML representation of the animation embedded as a js object as
            produced with the `.HTMLWriter`.
        """
        if fps is None and hasattr(self, '_interval'):
            # Convert interval in ms to frames per second
            fps = 1000 / self._interval

        # If we're not given a default mode, choose one base on the value of
        # the _repeat attribute
        if default_mode is None:
            default_mode = 'loop' if getattr(self, '_repeat',
                                             False) else 'once'

        if not hasattr(self, "_html_representation"):
            # Can't open a NamedTemporaryFile twice on Windows, so use a
            # TemporaryDirectory instead.
            with TemporaryDirectory() as tmpdir:
                path = Path(tmpdir, "temp.html")
                writer = animation.HTMLWriter(fps=fps,
                                    embed_frames=embed_frames,
                                    default_mode=default_mode)
                self.save(str(path), writer=writer, *args, **kwargs)
                self._html_representation = path.read_text()

        return self._html_representation


In [ ]:



ranks = np.array(list(range(20)) + list(range(20,min(height, width), 20)) + [min(height, width)]) 
percentages = (height * ranks + width+ranks + ranks)/(width * height) * 100
number_of_frames = len(percentages)

def animateCompression(i, U, Sigma, V_t, progress_callback=None):
    """
    This function takes the precomputed SVD matrices (U, Sigma, V_t)
    The rank used is the i-th rank of the global list `ranks`.
    """
    r = ranks[i]

    # truncate the SVD and reconstruct D
    D_prime = U[:, :r] @ Sigma[:r, :r] @ V_t[:r,:]

    # clear axs and plot reconstructed image
    for ax in [axl, axr] + list(axs):
        ax.cla()
    plot_face(D_prime, ax=axl)
    axl.set_title(f"Reconstructed Image")
    axr.set_title(f"Compression percentage")
    axr.plot(ranks[:i], percentages[:i], 'b', lw=2)
    axr.set_xlim((ranks[0],ranks[-1]))
    axr.set_ylim((percentages[0], percentages[-1]))
    A = np.zeros((height, width))
    z = A[:height, :r]
    if r > 0:
        axs[1].matshow(A[:height, :r])
        axs[2].matshow(A[:r, :r])
        axs[3].matshow(A[:r, :width])
    axs[0].matshow(A)
    if progress_callback:
        progress_callback()
    


fig = plt.figure(layout="constrained")
subfigs = fig.subfigures(2, 1, wspace=0.07, width_ratios=[[1.5, 1.]])
axl, axr = subfigs[0].subplots(1, 2)
axs = subfigs[1].subplots( ncols=4, sharex='all', sharey='all')



# # number_of_frames = 8
# anim = NewFuncAnimation(fig, partial(animateCompression, U=U, Sigma=Sigma, V_t=V_t), frames=number_of_frames, interval=200, blit=False)
# plt.close()
# from tqdm.notebook import tqdm
# with tqdm(unit='frames', total=number_of_frames, leave=False) as pbar:
#     html = anim.to_jshtml(progress_callback = lambda i, n: pbar.update())
#     HTML(html)



# number_of_frames = 8
from tqdm.notebook import tqdm
with tqdm(unit='frames', total=number_of_frames, leave=False) as pbar:
    anim = animation.FuncAnima...tion(fig, partial(animateCompression, U=U, Sigma=Sigma, V_t=V_t, progress_callback=pbar.update), frames=number_of_frames, interval=200, blit=False)
    # plt.close()
    html = anim.to_jshtml()
HTML(html)



In [ ]:

new_image, width, height = load_image(5)
U,sigma,V_t = np.linalg.svd(new_image, full_matrices=False)
Sigma = np.diag(sigma)


ranks = np.array(list(range(20)) + list(range(20,min(height, width), 20)) + [min(height, width)]) 
percentages = (height * ranks + width+ranks + ranks)/(width * height) * 100
number_of_frames = len(percentages)

def animateCompression(i, U, Sigma, V_t, progress_callback=None):
    """
    This function takes the precomputed SVD matrices (U, Sigma, V_t)
    The rank used is the i-th rank of the global list `ranks`.
    """
    r = ranks[i]

    # truncate the SVD and reconstruct D
    D_prime = U[:, :r] @ Sigma[:r, :r] @ V_t[:r,:]

    # clear axs and plot reconstructed image
    for ax in [axl, axr] + list(axs):
        ax.cla()
    plot_face(D_prime, ax=axl)
    axl.set_title(f"Reconstructed Image")
    axr.set_title(f"Compression percentage")
    axr.plot(ranks[:i], percentages[:i], 'b', lw=2)
    axr.set_xlim((ranks[0],ranks[-1]))
    axr.set_ylim((percentages[0], percentages[-1]))
    A = np.zeros((height, width))
    z = A[:height, :r]
    if r > 0:
        axs[1].matshow(A[:height, :r])
        axs[2].matshow(A[:r, :r])
        axs[3].matshow(A[:r, :width])
    axs[0].matshow(A)
    if progress_callback:
        progress_callback()
    


fig = plt.figure(layout="constrained")
subfigs = fig.subfigures(2, 1, wspace=0.07, width_ratios=[[1.5, 1.]])
axl, axr = subfigs[0].subplots(1, 2)
axs = subfigs[1].subplots( ncols=4, sharex='all', sharey='all')



# # number_of_frames = 8
# anim = NewFuncAnimation(fig, partial(animateCompression, U=U, Sigma=Sigma, V_t=V_t), frames=number_of_frames, interval=200, blit=False)
# plt.close()
# from tqdm.notebook import tqdm
# with tqdm(unit='frames', total=number_of_frames, leave=False) as pbar:
#     html = anim.to_jshtml(progress_callback = lambda i, n: pbar.update())
#     HTML(html)



# number_of_frames = 8
from tqdm.notebook import tqdm
with tqdm(unit='frames', total=number_of_frames, leave=False) as pbar:
    anim = animation.FuncAnimation(fig, partial(animateCompression, U=U, Sigma=Sigma, V_t=V_t, progress_callback=pbar.update), frames=number_of_frames, interval=200, blit=False)
    # plt.close()
    html = anim.to_jshtml()
HTML(html)

